# NeRF Results Visualization

This notebook renders novel views, creates 360° videos, and visualizes the 3D reconstruction results.

In [ ]:
import sys
from pathlib import Path
import torch
import matplotlib.pyplot as plt
import numpy as np

sys.path.insert(0, str(Path('..').resolve()))

from src.models import NeRF
from src.data import NeRFDataset
from src.rendering.volume_renderer import VolumeRenderer
from src.rendering import generate_rays
from src.utils import compute_psnr, save_image

## 1. Load Trained Model

In [ ]:
# Load checkpoint
CHECKPOINT_PATH = '../outputs/checkpoints/checkpoint_step_100000.pth'  # Update with your checkpoint
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Create models
model_coarse = NeRF().to(device)
model_fine = NeRF().to(device)

# Load weights
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
model_coarse.load_state_dict(checkpoint['model_coarse_state_dict'])
model_fine.load_state_dict(checkpoint['model_fine_state_dict'])

model_coarse.eval()
model_fine.eval()

print(f"Loaded checkpoint from step {checkpoint['global_step']}")

## 2. Render Novel Views

In [ ]:
# Load test dataset
DATA_DIR = '../data/nerf_synthetic/lego'
test_dataset = NeRFDataset(DATA_DIR, split='test', image_scale=0.5)

# Render a test image
test_sample = test_dataset[0]
pose = test_sample['pose'].to(device)
focal = test_sample['focal'].to(device)
near = test_sample['near'].to(device)
far = test_sample['far'].to(device)
gt_image = test_sample['image']

height, width = gt_image.shape[:2]

# Generate rays
ray_origins, ray_directions = generate_rays(
    height, width, focal.item(), pose, near.item(), far.item()
)

# Render (using fine model)
with torch.no_grad():
    rgb_image, depth_map = VolumeRenderer.render_image(
        model_fine,
        ray_origins.to(device),
        ray_directions.to(device),
        near.item(),
        far.item(),
        num_samples=128,
        chunk_size=4096,
    )

# Compute PSNR
psnr = compute_psnr(rgb_image.cpu(), gt_image)
print(f"PSNR: {psnr:.2f} dB")

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(gt_image.numpy())
axes[0].set_title('Ground Truth')
axes[0].axis('off')

axes[1].imshow(rgb_image.cpu().numpy())
axes[1].set_title(f'Rendered (PSNR: {psnr:.2f} dB)')
axes[1].axis('off')

axes[2].imshow(depth_map.cpu().squeeze().numpy(), cmap='jet')
axes[2].set_title('Depth Map')
axes[2].axis('off')

plt.tight_layout()
plt.show()

## 3. Create 360° Rotation Video

In [ ]:
# Generate poses for 360° rotation
num_frames = 60
radius = 4.0

render_poses = []
for i in range(num_frames):
    angle = 2 * np.pi * i / num_frames
    
    # Camera position on circle
    x = radius * np.cos(angle)
    z = radius * np.sin(angle)
    y = 0.0
    
    # Look at origin
    forward = -np.array([x, y, z])
    forward = forward / np.linalg.norm(forward)
    
    # Create pose matrix
    up = np.array([0, 1, 0])
    right = np.cross(forward, up)
    up = np.cross(right, forward)
    
    pose = np.eye(4)
    pose[:3, 0] = right
    pose[:3, 1] = up
    pose[:3, 2] = -forward
    pose[:3, 3] = [x, y, z]
    
    render_poses.append(pose)

# Render frames
frames = []
for pose in render_poses:
    pose_tensor = torch.from_numpy(pose).float().to(device)
    
    ray_origins, ray_directions = generate_rays(
        height, width, focal.item(), pose_tensor, near.item(), far.item()
    )
    
    with torch.no_grad():
        rgb_image, _ = VolumeRenderer.render_image(
            model_fine,
            ray_origins.to(device),
            ray_directions.to(device),
            near.item(),
            far.item(),
            num_samples=128,
            chunk_size=4096,
        )
    
    frames.append(rgb_image.cpu())
    print(f"Rendered frame {len(frames)}/{num_frames}")

# Save video
from src.utils import create_video
create_video(frames, '../outputs/360_rotation.mp4', fps=30)
print("Video saved!")